# 05 - Direction D: the agent uses the search tool and decides for itself

**Idea.** A Gemini agent gets the customer message and one tool, `search_kb` (the production retrieval, **no score gate**).
It may search up to a budget of times, then must decide: `ANSWER` (with citations), `ASK_CUSTOMER` or `NOT_IN_KB`.
A deterministic **validator** then checks grounding: an `ANSWER` with no citations, or citing a passage that was never returned
by one of this run's searches, is downgraded to `NOT_IN_KB`. The decision we score is `final_decision` (after the validator).

**Variants** (all cached under `cache/d_runs/<variant>/`):

| variant | scores shown to the agent | search budget | passages per search |
|---|---|---|---|
| `d_hidden_b3` (main) | hidden | 3 | 5 |
| `d_shown_b3` | shown | 3 | 5 |
| `d_hidden_b1` | hidden | 1 | 5 |
| `d_hidden_b3_k3` | hidden | 3 | 3 |
| `d_hidden_b3_k10` | hidden | 3 | 10 |

**Scoring.** Gold label: ANSWERABLE = the judged pool has a grade-2 passage. `did answer` = `final_decision == ANSWER`.
Answer-class F1, false-answer rate (unanswerable queries answered; also shown for ADJ/OOD only) and false-refusal rate
(answerable queries not answered). Citation correctness: is the cited passage graded 2?

| | |
|---|---|
| **Required caches** | `cache/d_runs/<variant>/*.json` - `uv run --env-file .env python -m experiments.llm.agent_d --variant d_hidden_b3` (repeat for each variant) |
| **Also required** | `cache/judgments.json` for labels and citation checks (`uv run python -m experiments.build_pools` then `uv run --env-file .env python -m experiments.llm.judge`). Run `build_pools` AFTER the agent runs so the agents' passages are in the judged pools. Optional: `cache/passages.json` (article slugs in the transcripts; made by `run_variants`). |

In [ ]:
import sys, pathlib, math, warnings
warnings.filterwarnings("ignore")
_here = pathlib.Path.cwd().resolve()
REPO = next(p for p in [_here, *_here.parents] if (p / "experiments" / "lab").is_dir())
sys.path[:0] = [str(REPO), str(REPO / "experiments")]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lab import notebook_utils as nu

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False})
print("cache directory:", nu.cache_dir())

In [ ]:
VARS = nu.d_variants()
D = {v: nu.load_d_runs(v) for v in VARS}
HAVE_D = bool(VARS)
Q = nu.load_queries()
J = nu.load_judgments()
HAVE_J = J is not None
QR = nu.qrels_from(J)
PAS = nu.load_passages()
labels = nu.gold_labels(Q, J)
if nu.need(HAVE_D, "d_runs/<variant>/*.json", nu.CMD_AGENT):
    print({v: len(r) for v, r in D.items()}, "queries per variant")
    print(nu.label_banner(labels))
    if not HAVE_J:
        display(nu.need(False, "judgments.json (citation correctness; real labels)", nu.CMD_POOLS + " ; " + nu.CMD_JUDGE))

## 1. Decisions: what did the agent answer, and what did the validator change?

In [ ]:
T = {}
if nu.need(HAVE_D, "d_runs", nu.CMD_AGENT):
    T = {v: nu.d_table(D[v], labels, QR if HAVE_J else None) for v in VARS}
    rows = []
    for v, t in T.items():
        rows.append({"variant": v, "queries": len(t), "ANSWER (agent)": int((t["decision"] == "ANSWER").sum()),
                     "ANSWER (final)": int((t["final"] == "ANSWER").sum()), "ASK_CUSTOMER": int((t["final"] == "ASK_CUSTOMER").sum()),
                     "NOT_IN_KB": int((t["final"] == "NOT_IN_KB").sum()), "downgraded by validator": int(t["downgraded"].sum()),
                     "runs with invalid citations": int((t["invalid_citations"] > 0).sum())})
    DEC = pd.DataFrame(rows).set_index("variant")
    display(DEC)

In [ ]:
if nu.need(bool(T), "d_runs", nu.CMD_AGENT):
    main = "d_hidden_b3" if "d_hidden_b3" in T else VARS[0]
    ct = pd.crosstab(T[main]["set"], T[main]["final"]).reindex(nu.SETS).fillna(0)
    ax = ct.plot(kind="bar", stacked=True, figsize=(7, 3.8), color={"ANSWER": "#2a7fbf", "ASK_CUSTOMER": "#e69f00", "NOT_IN_KB": "#999999"})
    ax.set_title(f"{main}: final decision by query set"); ax.set_xlabel("query set"); ax.set_ylabel("queries"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 2. Answer / refuse quality per variant

Bootstrap 95% intervals resample the queries (with only ~89 queries they are wide: read differences between variants
with care, section 5 tests them formally).

In [ ]:
def d_score(t):
    lab = t["label"].dropna().astype(bool)
    did = (t.loc[lab.index, "final"] == "ANSWER")
    m = nu.decision_metrics(lab.tolist(), did.tolist())
    ci = nu.bootstrap_ci(nu.f1_stat(lab.tolist(), did.tolist()), len(lab))
    un = ~lab
    far_ci = nu.bootstrap_ci(nu.rate_stat(did.tolist(), un.tolist()), len(lab))
    frr_ci = nu.bootstrap_ci(nu.rate_stat((~did).tolist(), lab.tolist()), len(lab))
    advers = t[t["set"].isin(["ADJ", "OOD"])]
    return {"answer F1": m["f1"], "F1 95% CI": f"[{ci[1]:.2f}, {ci[2]:.2f}]", "precision": m["precision"], "recall": m["recall"],
            "false-answer rate (unanswerable)": m["false_answer_rate"], "FAR 95% CI": f"[{far_ci[1]:.2f}, {far_ci[2]:.2f}]",
            "false-answer rate ADJ+OOD": float((advers["final"] == "ANSWER").mean()) if len(advers) else math.nan,
            "false-refusal rate (answerable)": m["false_refusal_rate"], "FRR 95% CI": f"[{frr_ci[1]:.2f}, {frr_ci[2]:.2f}]", "labelled queries": len(lab)}

if nu.need(bool(T), "d_runs", nu.CMD_AGENT):
    SCORE = pd.DataFrame({v: d_score(t) for v, t in T.items()}).T
    print(nu.label_banner(labels))
    display(SCORE)

## 3. Citations and grounding

For every run whose final decision is `ANSWER`: were the cited passages actually graded 2 ("directly answers")?
`all cited grade 2` is strict, `any cited grade 2` lenient, `citation precision` is the share of individual citations graded 2.

In [ ]:
if nu.need(bool(T) and HAVE_J, "d_runs + judgments.json", nu.CMD_POOLS + " ; " + nu.CMD_JUDGE):
    rows = []
    for v, t in T.items():
        a = t[t["final"] == "ANSWER"]
        rows.append({"variant": v, "answers": len(a), "all cited grade 2": a["cites_all_grade2"].dropna().astype(float).mean(),
                     "any cited grade 2": a["cites_any_grade2"].dropna().astype(float).mean(), "citation precision": a["cite_precision"].mean(),
                     "answers on unanswerable queries": int((a["label"] == False).sum()),  # noqa: E712
                     "avg citations / answer": a["n_cites"].mean()})
    CIT = pd.DataFrame(rows).set_index("variant")
    display(CIT)

## 4. Searches, tokens and latency

In [ ]:
if nu.need(bool(T), "d_runs", nu.CMD_AGENT):
    fig, ax = plt.subplots(figsize=(7, 3.6))
    width = 0.8 / len(T)
    for i, (v, t) in enumerate(T.items()):
        vc = t["n_searches"].value_counts().reindex(range(0, 4), fill_value=0)
        ax.bar(np.arange(4) + i * width, vc.to_numpy(), width, label=v)
    ax.set_xticks(np.arange(4) + 0.4 - width / 2); ax.set_xticklabels(["0", "1", "2", "3"])
    ax.set_title("Number of searches the agent used (per query)"); ax.set_xlabel("searches"); ax.set_ylabel("queries"); ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()
    rows = [{"variant": v, "mean searches": t["n_searches"].mean(), "LLM requests / query": t["requests"].mean(),
             "input tokens / query": t["in_tokens"].mean(), "output tokens / query": t["out_tokens"].mean(),
             "total tokens / query": (t["in_tokens"] + t["out_tokens"]).mean(),
             "p50 latency s": nu.p50(t["latency_s"]), "p90 latency s": float(np.nanpercentile(t["latency_s"].astype(float), 90))} for v, t in T.items()]
    COST = pd.DataFrame(rows).set_index("variant")
    display(COST)

## 5. Variant comparisons (paired bootstrap on answer F1)

- scores **shown vs hidden** to the agent (`d_shown_b3` vs `d_hidden_b3`)
- search **budget** 1 vs 3 (`d_hidden_b1` vs `d_hidden_b3`)
- **top_k** 3 / 5 / 10 (`d_hidden_b3_k3`, `d_hidden_b3`, `d_hidden_b3_k10`)

A difference whose interval contains 0 is not evidence of a real effect.

In [ ]:
def did_vec(t, ids):
    return (t.loc[ids, "final"] == "ANSWER").to_numpy()

PAIRS = [("scores shown vs hidden", "d_shown_b3", "d_hidden_b3"), ("budget 1 vs 3", "d_hidden_b1", "d_hidden_b3"),
         ("top_k 3 vs 5", "d_hidden_b3_k3", "d_hidden_b3"), ("top_k 10 vs 5", "d_hidden_b3_k10", "d_hidden_b3")]
if nu.need(bool(T), "d_runs", nu.CMD_AGENT):
    rows = []
    for name, a, b in PAIRS:
        if a not in T or b not in T:
            rows.append({"comparison": name, "A": a, "B": b, "note": "variant not run yet: " + nu.CMD_AGENT.replace("d_hidden_b3", a if a not in T else b)})
            continue
        lab = T[a]["label"].dropna().astype(bool)
        ids = lab.index.intersection(T[b].index)
        lab = lab.loc[ids]
        d = nu.bootstrap_diff(nu.f1_stat(lab.to_numpy(), did_vec(T[a], ids)), nu.f1_stat(lab.to_numpy(), did_vec(T[b], ids)), len(ids))
        rows.append({"comparison": name, "A": a, "B": b, "F1 A - B": d["diff"], "CI low": d["lo"], "CI high": d["hi"], "zero inside CI": d["lo"] <= 0 <= d["hi"], "queries": len(ids)})
    CMP = pd.DataFrame(rows).set_index("comparison")
    display(CMP)

In [ ]:
if nu.need(bool(T) and "SCORE" in globals(), "d_runs", nu.CMD_AGENT):
    fig, ax = plt.subplots(figsize=(9, 3.8))
    x = np.arange(len(SCORE))
    for k, (col, c) in enumerate((("answer F1", "#2a7fbf"), ("false-answer rate (unanswerable)", "#d9534f"), ("false-refusal rate (answerable)", "#e69f00"))):
        ax.bar(x + k * 0.27, SCORE[col].astype(float), 0.27, label=col, color=c)
    ax.set_xticks(x + 0.27); ax.set_xticklabels(SCORE.index, rotation=15, fontsize=8)
    ax.set_title("Direction D variants: answer F1 and error rates"); ax.set_ylabel("rate / F1"); ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()

## 6. Eight example transcripts (main variant)

Chosen by rule: 2 correct answers, 2 false answers (answered but unanswerable), 2 false refusals (answerable but not answered),
1 validator downgrade, 1 ASK_CUSTOMER. If a category is empty the remaining slots are filled with other runs.

In [ ]:
def show(qid, rec, t):
    print("=" * 100)
    print(f"{qid} [{t.loc[qid, 'set']}] gold label: {nu.label_name(t.loc[qid, 'label'])}   ->  decision {rec['decision']}, FINAL {rec['final_decision']}")
    print("QUERY:", " ".join(Q.loc[qid, "text"].split())[:300])
    for i, s in enumerate(rec["searches"], 1):
        hits = ", ".join(f"{PAS.get(p, {}).get('slug', p[:8])}" + (f" ({sc:.1f}" + (f", grade {QR[qid].get(p)}" if qid in QR else "") + ")") for p, sc in list(zip(s["passage_ids"], s["scores"]))[:5])
        print(f"  search {i}: '{s['query']}' -> {hits}")
    if rec["citations"]:
        print("  cites:", [(PAS.get(c, {}).get("slug", c[:8]), QR.get(qid, {}).get(c)) for c in rec["citations"]], "invalid:", rec["invalid_citations"])
    print("  answer:", " ".join(rec["answer_text"].split())[:350])

if nu.need(bool(T), "d_runs", nu.CMD_AGENT):
    t = T[main]; rng = np.random.default_rng(1)
    lab, fin = t["label"], t["final"]
    cats = [("correct answer", t.index[(lab == True) & (fin == "ANSWER")], 2), ("false answer", t.index[(lab == False) & (fin == "ANSWER")], 2),  # noqa: E712
            ("false refusal", t.index[(lab == True) & (fin != "ANSWER")], 2), ("validator downgrade", t.index[t["downgraded"]], 1),  # noqa: E712
            ("ASK_CUSTOMER", t.index[fin == "ASK_CUSTOMER"], 1)]
    chosen = []
    for _, ids, k in cats:
        ids = [i for i in ids if i not in chosen]
        chosen += list(rng.choice(ids, size=min(k, len(ids)), replace=False)) if ids else []
    chosen += [i for i in t.index if i not in chosen][: 8 - len(chosen)]
    for qid in chosen[:8]:
        show(qid, D[main][qid], t)

# RESULT TABLE - Direction D variants

One row per variant: decision quality against the gold labels and what it costs per query.

In [ ]:
if nu.need(bool(T) and "SCORE" in globals(), "d_runs", nu.CMD_AGENT):
    RES = SCORE[["answer F1", "F1 95% CI", "false-answer rate (unanswerable)", "false-answer rate ADJ+OOD", "false-refusal rate (answerable)"]].copy()
    RES["downgraded by validator"] = DEC["downgraded by validator"]
    RES["LLM requests/query"] = COST["LLM requests / query"]; RES["tokens/query"] = COST["total tokens / query"]; RES["p50 latency s"] = COST["p50 latency s"]
    if "CIT" in globals():
        RES["all-cited grade 2"] = CIT["all cited grade 2"]
    RES = RES.sort_values("answer F1", ascending=False)
    display(RES)
    print(nu.label_banner(labels))
    print(f"BEST D VARIANT by F1: {RES.index[0]} (F1 {float(RES.iloc[0]['answer F1']):.3f}).")